# Thermal budget

The SOA budget, and a burst planned against it.

In [1]:
SIMULATED = True          # False, and PORT, at the bench
PORT = 'COM4'

In [2]:
from coaxial import Coaxial63100

device = Coaxial63100(port=PORT, simulated_device=SIMULATED).open()
print(device)

<Coaxial63100 Simulated SIMULATED>


The board never calls a reading good: it reports the margin and acts at a ceiling by dropping MOE, the same path the break uses. The ceilings live in the calibration record (invariant 10).

In [3]:
thermal = device.thermal
st = thermal.state()
print('NTC', st['ntc'], ' ambient', st['ambient'], ' settled', st['settled'],
      ' every', st['sample_every_s'], 's')
for node, celsius in st['nodes'].items():
    print('%-12s %6.2f C' % (node, celsius))

NTC 37.0  ambient 20.0  settled True  every 5.0 s
driver_u      32.00 C
driver_v      32.00 C
driver_w      32.00 C
phase_u       31.40 C
phase_v       31.40 C
phase_w       31.40 C
mcu           46.00 C
regulators    39.00 C
afe           36.90 C
board         31.00 C


`used` is a fraction, 0 at ambient and 1 at the node's ceiling: a temperature cannot say how close a part is without its limit beside it, so the board sends the fraction and keeps degrees on `state()`. The ceilings themselves are the record's - an uncalibrated board holds none, and a node with no ceiling is not judged.

In [4]:
from coaxial.thermal import ALL_NODES

cal = device.calibration.read()
budget = thermal.budget()
limits = dict(zip(ALL_NODES, cal['soa_limit_c']))
print('ceilings in the record:', len(limits), ' throttle at', cal['soa_throttle_at'])
for node in ALL_NODES:
    print('%-12s %6.2f C  used %5.1f %%  limit %s'
          % (node, st['nodes'][node], 100.0 * budget['used'][node],
             '%.1f C' % limits[node] if node in limits else 'none in the record'))
print('worst', budget['worst_node'], ' seconds_to_limit', budget['seconds_to_limit'],
      ' throttling', budget['throttling'], ' tripped', budget['tripped'],
      ' trips', budget['trips'])

ceilings in the record: 0  throttle at 0.0
driver_u      32.00 C  used  11.4 %  limit none in the record
driver_v      32.00 C  used  11.4 %  limit none in the record
driver_w      32.00 C  used  11.4 %  limit none in the record
phase_u       31.40 C  used  10.9 %  limit none in the record
phase_v       31.40 C  used  10.9 %  limit none in the record
phase_w       31.40 C  used  10.9 %  limit none in the record
mcu           46.00 C  used  24.8 %  limit none in the record
regulators    39.00 C  used  18.1 %  limit none in the record
afe           36.90 C  used  16.1 %  limit none in the record
board         31.00 C  used  12.9 %  limit none in the record
worst mcu  seconds_to_limit None  throttling False  tripped False  trips 0


A burst planned against the network in `coaxial.thermal`: a node's rise over the board is `P * to_board`, reached on its own time constant `capacity * to_board`, while the board itself rises on 6.8 minutes. With a ceiling in the record the same arithmetic gives the seconds to it.

In [5]:
import math
from coaxial.thermal import CFG, tau_minutes

def rise_after(node, watts, seconds):
    tau = CFG['capacity'][node] * CFG['to_board'][node]
    return watts * CFG['to_board'][node] * (1.0 - math.exp(-seconds / tau))

def seconds_to(node, watts, now_c, ceiling_c):
    tau = CFG['capacity'][node] * CFG['to_board'][node]
    top = now_c + watts * CFG['to_board'][node]
    if top <= ceiling_c:
        return None
    return -tau * math.log(1.0 - (ceiling_c - now_c) / (top - now_c))

print('board tau %.1f min' % tau_minutes())
for node in ('phase_u', 'driver_u', 'mcu'):
    tau = CFG['capacity'][node] * CFG['to_board'][node]
    print('%-9s %5.1f K/W, tau %.2f s' % (node, CFG['to_board'][node], tau))
    for watts in (5.0, 15.0, 35.0):
        line = ('%6.1f W: +%5.1f K after 100 ms, +%5.1f K steady'
                % (watts, rise_after(node, watts, 0.1), watts * CFG['to_board'][node]))
        if node in limits and cal['soa_throttle_at']:
            ceiling = limits[node] * cal['soa_throttle_at']
            t = seconds_to(node, watts, st['nodes'][node], ceiling)
            line += ('  throttle point %.1f C: %s'
                     % (ceiling, 'never' if t is None else '%.2f s' % t))
        print('   ' + line)

board tau 6.8 min
phase_u    45.6 K/W, tau 18.24 s
      5.0 W: +  1.2 K after 100 ms, +228.0 K steady
     15.0 W: +  3.7 K after 100 ms, +684.0 K steady
     35.0 W: +  8.7 K after 100 ms, +1596.0 K steady
driver_u   45.6 K/W, tau 5.32 s
      5.0 W: +  4.2 K after 100 ms, +228.0 K steady
     15.0 W: + 12.7 K after 100 ms, +684.0 K steady
     35.0 W: + 29.7 K after 100 ms, +1596.0 K steady
mcu        22.5 K/W, tau 20.25 s
      5.0 W: +  0.6 K after 100 ms, +112.5 K steady
     15.0 W: +  1.7 K after 100 ms, +337.5 K steady
     35.0 W: +  3.9 K after 100 ms, +787.5 K steady


`set_sample` is how often the observer borrows AFE_ON for an NTC reading when nothing else holds the rail.

In [6]:
print(thermal.set_sample(30.0, settle_s=0.5))
print(thermal.state()['sample_every_s'])
device.close()

True
30.0


## Conclusions

In [7]:
print('measured         NTC %s C' % st['ntc'])
print('model says       NTC %.2f C, error %s' % (st['expected_ntc'], st['error']))
print('ambient          %.1f C, settled %s, %d integration steps'
      % (st['ambient'], st['settled'], st['steps']))
print('other dies       afe %s C, mcu %s C, seen %.1f s ago'
      % (st['afe'], st['mcu'], st['seen_s_ago']))
print('worst node       %s at %.1f %% of its ceiling'
      % (budget['worst_node'], 100.0 * budget['worst']))
print('acting           throttling %s, tripped %s, trips %d'
      % (budget['throttling'], budget['tripped'], budget['trips']))

measured         NTC 37.0 C
model says       NTC 37.00 C, error 0.0
ambient          20.0 C, settled True, 1200 integration steps
other dies       afe 36.9 C, mcu 46.0 C, seen 0.4 s ago
worst node       mcu at 24.8 % of its ceiling
acting           throttling False, tripped False, trips 0


One measurement and nine estimates. The NTC is the only thermometer that sees the power stage, and `error` - the model's expected NTC minus the measured one - is the only number that says whether the parameters hold.

`ntc` is None while AFE_ON is low, which is when the drivers have supply and switching is possible: the sensor and the drivers share one switch. The model then runs open on power and time.

This is the narrow exception to invariant 10. The board never calls a reading good - it reports the margin and *acts*, dropping MOE at a ceiling by the path the break uses. The ceilings live in the calibration record, a limit it was given rather than invented, and a node with none is not judged.

`set_sample` is how often the observer borrows AFE_ON when nothing else holds the rail; while another subsystem holds it the NTC is read every step, and an acquire is refused while the stage is armed.